# Supplementary Methods 8-11: statistics

Supplementary Methods 8-11 describe the clinical, regional and cross-omics analyses, the histidine-boundary
analyses, the statistical framework with its testing-family map, and computational provenance. This notebook
recomputes the statistics and counts these sections state, and checks every Benjamini-Hochberg family of the
testing-family map (v28d line 893) by recomputing each adjusted q value within exactly that family.
Run from the repository root with `python paper.py build supplementary_methods_8_11`.

It reads `source_data/supplementary_methods_8_11/` and tables owned by other units (`Fig5`, `Fig6`, `SFig10`,
`SFig11`, `SFig14`, `STab3`, `STab5`), named in each section. There are no figure panels in this unit.

In [ ]:
from pathlib import Path
import sys
UNIT = Path.cwd()                 # the notebook runs with its own directory as cwd
REPO = UNIT.parents[1]
sys.path[:0] = [str(REPO), str(UNIT)]
SD = REPO / "source_data" / UNIT.name
OUT = REPO / "build" / UNIT.name   # gitignored
OUT.mkdir(parents=True, exist_ok=True)
from style import fl_style as fl
import stats
import pandas as pd
fl.init()
SDR = REPO / "source_data"         # stats functions take the source_data root and read other units' tables by path
pd.set_option("display.width", 200, "display.max_columns", 30)
print(stats.software())

## CapScan regional human-assigned share

"Two-sided paired sign-flip tests enumerated all 32,768 possible sign patterns across the fifteen participants, with
Benjamini–Hochberg correction across the four regional contrasts" and "95% intervals from 2,000 participant-bootstrap
resamples" (v28d line 843). n = 15 participants, paired (region and stool within participant). Recomputed from the
participant-region origin fractions (`source_data/Fig5/origin_fraction_cells.tsv`) and compared with the producer's
`four_region_tests.tsv`. The bootstrap seed (20261003) is not stated in the text.

In [ ]:
rg = stats.stat_regional_human_share(SDR)
print("sign patterns:", rg["patterns"], "| contrasts:", rg["contrasts"], "| max |difference| to producer table:", rg["max_abs_error"])
rg["tests"]

## CapScan own-stool permutation test

"Significance was assessed by permuting participant labels of the stool profiles 99,999 times (seed 20260910) ...
two-sided plus-one permutation P value" (v28d line 849); "with every participant closer to their own stool and
P≤1 × 10⁻⁵" (v28d line 847). n = 15 participants, 75 profiles. Recomputed from the 15 x 15 capsule-to-stool distance
matrices (`source_data/supplementary_methods_8_11/own_stool_distance_matrices.tsv`), which reproduce the producer's
own and other distances exactly. No permuted value was as extreme as observed, so P equals its minimum 1/(N+1); the
permutation seed cannot be confirmed from the output. The 95% interval (2,000 participant resamples, seed 20261003)
is not described in the text.

In [ ]:
ow = stats.stat_own_stool(SDR, UNIT.name)
print("max |difference| to producer similarity_summary.tsv:", ow["max_abs_error"])
ow["tests"]

## Cross-omics profile concordance: families

PREDICT: "Benjamini–Hochberg correction was applied across the 28 PREDICT profile-level comparisons" (v28d line 827),
77 patients, 1,999 patient-label permutations. CapScan: "Benjamini–Hochberg correction was applied jointly across all 16
functional-profile comparisons" (v28d line 835), 13 participants, 65 profiles, whole-participant block permutations.
Native profiles: "Benjamini–Hochberg families contained ten or six comparisons" (v28d line 815). Tables:
`source_data/SFig11/functional_correlations_PREDICT.tsv`, `source_data/SFig11/native_correlations_PREDICT.tsv`,
`source_data/Fig5/functional_correlations.tsv`, `source_data/Fig5/native_correlations.tsv` (CapScan on the 347-acquisition
roster). The correlations and permutation P values themselves are recomputed by `figures/Fig5` and `figures/SFig11`.

In [ ]:
cr = stats.stat_crossomics(SDR)
print("functional rows:", cr["functional_rows"], "| native rows:", cr["native_rows"], "| max |q error|:", cr["max_abs_q_error"])
print("CapScan mixed sensitivity: non-DNA tests", cr["mixed_nondna_tests"], "on", cr["mixed_nondna_participants"], "participants;",
      "DNA tests", cr["mixed_dna_tests"], "on", cr["mixed_dna_participants"])
print("CapScan functional comparisons changing classification at q<0.05 across filters and populations:", cr["classification_changes"])
display(cr["functional_families"]); display(cr["native_families"])

## Controlled distance sensitivity

"Every method/adjustment/input variant used the same design-preserving 1,999 permutations and its own ten- or
six-comparison Benjamini–Hochberg family ... All 176 estimates ... were retained" (v28d line 821). PREDICT 77 patients;
CapScan 13 participants, 65 profiles. Table: `source_data/SFig11/controlled_176_comparisons.tsv`.

In [ ]:
co = stats.stat_controlled(SDR)
print("estimates:", co["rows"], "| family sizes:", co["family_sizes"], "| permutations:", co["permutations"], "| max |q error|:", co["max_abs_q_error"])
co["families"]

## Individual-function correlations

"Benjamini–Hochberg correction was applied separately within each of the nine representation-by-layer-pair families"
(v28d line 829). Statistical unit patient; at least 35 patients per test. Table:
`source_data/STab5/crossomics_same_function_correlations.tsv`.

In [ ]:
ind = stats.stat_individual_functions(SDR)
print("tests:", ind["rows"], "| families:", ind["n_families"], "| minimum patients:", ind["min_patients"], "| max |q error|:", ind["max_abs_q_error"])
ind["families"]

## EDS screen

"Benjamini–Hochberg correction was applied separately within each layer, normalization and model to all eligible
features" (v28d line 799); features required positive finite measurements in at least 43 patients. Table:
`source_data/SFig10/all_EDS_associations.tsv`.

In [ ]:
sc = stats.stat_eds_screen(SDR)
print("rows:", sc["rows"], "| minimum n among tested features:", sc["min_n_tested"], "| max |q error|:", sc["max_abs_q_error"])
sc["families"]

## Held-out EDS prediction

"Ridge penalties ... were selected by mean squared error in three inner folds within each of five shuffled
patient-separated outer folds. Three outer split seeds (20260915–20260917) were retained ... 199 label permutations"
(v28d line 803); "Benjamini–Hochberg adjustment across eight grouping-by-model tests ... Two thousand patient
resamples of fixed held-out predictions provided intervals" (v28d line 805). n = 85 patients. The held-out rho,
permutation P, q and both bootstrap intervals are recomputed from the stored predictions and permutation nulls; the
outer folds are regenerated with `KFold(5, shuffle=True, random_state=seed)`. The bootstrap seed (20260918) and the
label-permutation seed (20260915) are not stated in the text.

In [ ]:
ed = stats.stat_eds_prediction(SDR, UNIT.name)
print({k: ed[k] for k in ["n", "n_tests", "permutations", "perm_err", "q_err", "rho_err", "ci_err", "inc_err", "fold_ok", "outer_folds", "inner_folds", "seeds", "alphas_selected"]})
ed["tests"][["grouping", "model", "rho", "permutation_p", "p_recomputed", "permutation_q_8", "q_recomputed", "rho_ci_low", "rho_ci_high"]]

## PREDICT antibiotic endpoints

"Human-assigned LFQ share and quantified protein-group depth formed one two-test Benjamini–Hochberg family, separately
for the primary definition and each sensitivity definition" (v28d line 795). n = 42 patients (primary). The paired
sign-flip tests and bootstrap intervals need controlled clinical inputs and are recomputed by `tables/STab3`; here the
family structure and the 200,000-draw Monte Carlo grid of the P values are checked
(`source_data/STab3/endpoints_by_definition.tsv`).

In [ ]:
ab = stats.stat_antibiotic(SDR)
print("family size:", ab["family_size"], "| P values on the 1/200,001 grid:", ab["on_grid"], "| patients:", ab["n_patients"], "| max |q error|:", ab["max_abs_q_error"])
ab["families"]

## CapScan leading species and Fig. 5e persistence

"Eligibility required a resolved leader in at least twelve of the fifteen participants" (v28d line 785) and "This
30.8% therefore refers only to the 24 displayed functions" (v28d line 787). Tables: `source_data/Fig5/contributor_cells.tsv`,
`source_data/Fig5/mosaic_donor_regions.tsv`.

In [ ]:
stats.stat_contributors(SDR)

## DNA runs and native sequence features

"CapScan retained thirteen participants with all five regions, using 245 DNA runs" (v28d line 809); "gene TPM values
from 301 completed metagenome runs" (v28d line 833); "5,391,526 encoded-sequence features had positive RNA abundance,
compared with 14,822,078 for DNA" (v28d line 827). Tables: `capscan_dna_runs.tsv`, `predict_native_feature_summary.tsv`
in this unit's source data.

In [ ]:
print(stats.stat_dna_runs(SDR, UNIT.name))
print(stats.stat_native_features(SDR, UNIT.name))

## Supplementary Methods 9: non-specific searches and histidine families

"All 331 CapScan and 306 PREDICT acquisitions were searched" with the standard settings (v28d line 855); the CapScan
count is 347 on the v28f roster. Histidine families (v28d lines 869, 873, 893) are counted and every q value is
recomputed within its family from `source_data/Fig6/`. P values of tests with n ≤ 16 lie on the exact 2^-n grid and
those of PREDICT tests with n = 93 on the 1/200,001 Monte Carlo grid.

In [ ]:
display(pd.DataFrame(stats.stat_nonspecific(SDR, UNIT.name)).T)
hi = stats.stat_histidine_families(SDR)
print("exact grid (n<=16):", hi["capscan_exact_grid"], "| Monte Carlo grid (n>16):", hi["predict_mc_grid"], "| PREDICT patients:", hi["predict_patients"], "| max |q error|:", hi["max_abs_q_error"])
hi["families"]

## Supplementary Methods 11: healthy-stool grouping counts

"... identical output tables for the 58 acquisitions, including 59,428 quantified protein groups and 1,909,973 quantified
peptide features" (v28d line 909). Values read from the production grouping receipts
(`hstool_grouping_receipt.tsv`); the receipt of the public-module parity check itself was not located (OPEN_ITEMS.md).

In [ ]:
stats.stat_hstool_grouping(SDR, UNIT.name)

## Checks

There are no quantitative panels in this unit, so there are no pixel checks. Every recomputed number is compared with
the `recomputed_value` column of `NUMBERS.tsv`, and the statistics table is regenerated and compared with `STATS.tsv`.

In [ ]:
import csv
print("pixel checks: none (text unit, no panels)")
R, D = stats.collect(SDR, UNIT.name)
print("producer agreement checks:", D["checks"])
want = list(csv.DictReader(open(UNIT / "NUMBERS.tsv", encoding="utf-8"), delimiter="\t"))
got = stats.number_rows(R)
assert len(want) == len(got)
for w, g in zip(want, got):
    assert (w["v28d_line"], w["printed_value"], w["recomputed_value"], w["status"]) == (g["v28d_line"], g["printed_value"], g["recomputed_value"], g["status"]), (w, g)
    print(f'{g["v28d_line"]:>4}  {g["printed_value"]:>26}  {g["recomputed_value"]:>22}  {g["status"]}')
srows = list(csv.DictReader(open(UNIT / "STATS.tsv", encoding="utf-8"), delimiter="\t"))
new = stats.stats_rows(D)
assert len(srows) == len(new) and all({k: a[k] for k in b} == b for a, b in zip(srows, new)), "STATS.tsv differs from the recomputed statistics"
print("STATS.tsv rows reproduced:", len(new))